In [ ]:
import json
import os
import geopandas as gpd
import momepy as mp
import networkx as nx

In [ ]:
FOLDEROOT = "../data/processed/"
FOLDER_OUT = FOLDEROOT + "bikenetkit/"
END_FOLDER = ["Nothing", "2021", "2026"]
BUFF_SIZE = 400

In [ ]:
gdf_edges = gpd.read_file(FOLDEROOT + "bikenet_edges.gpkg")

# Make initial edges geojson separated

In [ ]:
def find_init_gdf(gdf_edges, end_folder):
    gdf = gdf_edges.copy()
    if end_folder == "2021":
        gdf["built"] = gdf["built_in"].apply(lambda x: 1 if x == "2021-01-01" else 0)
    elif end_folder == "2026":
        gdf["built"] = gdf["built_in"].apply(lambda x: 1 if x != "No" else 0)
    elif end_folder == "Nothing":
        G = mp.gdf_to_nx(gdf, integer_labels=False, preserve_index=True)
        closeness = nx.closeness_centrality(G, distance="length")
        edge_closeness = {
            edge: (closeness[edge[0]] + closeness[edge[1]]) / 2
            for edge in G.edges
            if (
                (G.edges[edge]["highway"] == "primary")
                & (G.edges[edge]["level"] == "primary")
                & (G.edges[edge]["built_in"] == "2021-01-01")
            )
        }
        choice = max(edge_closeness, key=edge_closeness.get)
        gdf["built"] = gdf.apply(
            lambda df: (
                1
                if (
                    ((df["from"] == str(choice[0])) & (df["to"] == str(choice[1])))
                    or ((df["to"] == str(choice[0])) & (df["from"] == str(choice[1])))
                )
                else 0
            ),
            axis=1,
        )
    gdf = gdf[gdf["built"] == 1].reset_index()[
        ["level", "highway", "length", "geometry"]
    ]
    gdf = gdf.rename(
        {"level": "bikenet_hierarchy", "highway": "road_hierarchy"}, axis=1
    )
    return gdf

In [ ]:
for end in END_FOLDER:
    folder_out = FOLDEROOT + f"bikenetkit/{end}/"
    if not os.path.exists(folder_out):
        os.makedirs(folder_out)
    gdf = find_init_gdf(gdf_edges, end)
    gdf.to_file(folder_out + f"init_{end}.geojson")

# Make real network steps

In [ ]:
gdf_timeline = gdf_edges.copy()

In [ ]:
gdf_timeline = (
    gdf_timeline[["built_in", "level", "highway", "length", "geometry"]]
    .sort_values(["built_in"], axis=0)
    .rename({"level": "bikenet_hierarchy", "highway": "road_hierarchy"}, axis=1)
)

In [ ]:
gdf_timeline.to_file(FOLDER_OUT + "real_growth.geojson")

# Make synthetic order steps

In [ ]:
foldername = FOLDEROOT + "Nothing/bs_400_betweenness/"
with open(foldername + "order_growth.json") as f:
    order_growth = json.load(f)
order_growth = [tuple((tuple(val[0]), tuple(val[1]), val[2])) for val in order_growth]
with open(foldername + "metrics_growth.json") as f:
    metrics_growth = json.load(f)

In [ ]:
with open(foldername + "metrics_growth.json") as f:
    metrics_growth = json.load(f)

In [ ]:
gdf_res = gdf_edges.copy()
gdf_res["key"] = gdf_edges.apply(
    lambda df: tuple(
        (
            tuple(
                [
                    float(val)
                    for val in df["from"]
                    .removeprefix("(")
                    .removesuffix(")")
                    .split(", ")
                ]
            ),
            tuple(
                [
                    float(val)
                    for val in df["to"].removeprefix("(").removesuffix(")").split(", ")
                ]
            ),
            0,
        )
    ),
    axis=1,
)
gdf_res = gdf_res[["key", "level", "highway", "length", "geometry"]].rename(
    {"level": "bikenet_hierarchy", "highway": "road_hierarchy"}, axis=1
)

In [ ]:
order_growth_dict = {val: idx for idx, val in enumerate(order_growth)}
order_growth_dict_reversed = {
    val: idx
    for idx, val in enumerate(tuple((val[1], val[0], val[2])) for val in order_growth)
}

In [ ]:
gdf_res["order"] = gdf_res["key"].map(order_growth_dict)
gdf_res["order_rev"] = gdf_res["key"].map(order_growth_dict_reversed)
gdf_res["order"] = gdf_res["order"].fillna(gdf_res["order_rev"])

In [ ]:
gdf_res = gdf_res[gdf_res["order"].notna()]

In [ ]:
gdf_res["order"] = gdf_res["order"].map(int)

In [ ]:
gdf_res = gdf_res.set_index("order").sort_index()

In [ ]:
gdf_res = gdf_res[["bikenet_hierarchy", "road_hierarchy", "length", "geometry"]]

In [ ]:
for key, arr in metrics_growth.items():
    gdf_res[key] = arr[1:]

In [ ]:
def init_gdf(gdf_edges, end_folder):
    gdf = gdf_edges.copy()
    if end_folder == "2021":
        gdf["built"] = gdf["built_in"].apply(lambda x: 1 if x == "2021-01-01" else 0)
    elif end_folder == "2026":
        gdf["built"] = gdf["built_in"].apply(lambda x: 1 if x != "No" else 0)
    elif end_folder == "Nothing":
        G = mp.gdf_to_nx(gdf, integer_labels=False, preserve_index=True)
        closeness = nx.closeness_centrality(G, distance="length")
        edge_closeness = {
            edge: (closeness[edge[0]] + closeness[edge[1]]) / 2
            for edge in G.edges
            if (
                (G.edges[edge]["highway"] == "primary")
                & (G.edges[edge]["level"] == "primary")
                & (G.edges[edge]["built_in"] == "2021-01-01")
            )
        }
        choice = max(edge_closeness, key=edge_closeness.get)
        gdf["built"] = gdf.apply(
            lambda df: (
                1
                if (
                    ((df["from"] == str(choice[0])) & (df["to"] == str(choice[1])))
                    or ((df["to"] == str(choice[0])) & (df["from"] == str(choice[1])))
                )
                else 0
            ),
            axis=1,
        )
    return gdf

In [ ]:
for end in END_FOLDER:
    folder_out = FOLDEROOT + f"bikenetkit/{end}/"
    gdf_res = init_gdf(gdf_edges, end)
    gdf_res["key"] = gdf_edges.apply(
        lambda df: tuple(
            (
                tuple(
                    [
                        float(val)
                        for val in df["from"]
                        .removeprefix("(")
                        .removesuffix(")")
                        .split(", ")
                    ]
                ),
                tuple(
                    [
                        float(val)
                        for val in df["to"]
                        .removeprefix("(")
                        .removesuffix(")")
                        .split(", ")
                    ]
                ),
                0,
            )
        ),
        axis=1,
    )
    gdf_res = gdf_res[
        ["key", "level", "highway", "length", "built", "geometry"]
    ].rename({"level": "bikenet_hierarchy", "highway": "road_hierarchy"}, axis=1)
    for met in [
        "betweenness",
        "closeness",
        "coverage",
        "directness",
        "dual_betweenness",
        "dual_closeness",
        "random",
        "road_hierarchy",
        "road_hierarchy_coverage",
        "road_hierarchy_directness",
    ]:
        gdf_order = gdf_res.copy()
        foldername = FOLDEROOT + end + f"/bs_{BUFF_SIZE}_{met}/"
        if met in ["coverage", "road_hierarchy", "random"]:
            foldername += met + "_000/"
        with open(foldername + "order_growth.json") as f:
            order_growth = json.load(f)
        order_growth = [
            tuple((tuple(val[0]), tuple(val[1]), val[2])) for val in order_growth
        ]
        with open(foldername + "metrics_growth.json") as f:
            metrics_growth = json.load(f)
        order_growth_dict = {val: idx + 1 for idx, val in enumerate(order_growth)}
        order_growth_dict_reversed = {
            val: idx + 1
            for idx, val in enumerate(
                tuple((val[1], val[0], val[2])) for val in order_growth
            )
        }
        gdf_order["order"] = gdf_order["key"].map(order_growth_dict)
        gdf_order["order"] = gdf_order.apply(
            lambda df: 0 if df["built"] == 1 else df["order"], axis=1
        )
        gdf_order["order_rev"] = gdf_order["key"].map(order_growth_dict_reversed)
        gdf_order["order"] = gdf_order["order"].fillna(gdf_order["order_rev"]).map(int)
        gdf_order = gdf_order.set_index("order").sort_index()
        gdf_order["total_length"] = [metrics_growth["xx"][0]] * len(
            gdf_res[gdf_res["built"] == 1]
        ) + metrics_growth["xx"][1:]
        gdf_order["total_coverage"] = [metrics_growth["coverage"][0]] * len(
            gdf_res[gdf_res["built"] == 1]
        ) + metrics_growth["coverage"][1:]
        gdf_order["total_directness"] = [metrics_growth["directness"][0]] * len(
            gdf_res[gdf_res["built"] == 1]
        ) + metrics_growth["directness"][1:]
        gdf_order = gdf_order[
            [
                "bikenet_hierarchy",
                "road_hierarchy",
                "length",
                "total_length",
                "total_coverage",
                "total_directness",
                "geometry",
            ]
        ]
        gdf_order.to_file(folder_out + f"{met}_order.geojson")

In [ ]:
for end in END_FOLDER:
    folder_out = FOLDEROOT + f"bikenetkit/{end}/"
    gdf_res = init_gdf(gdf_edges, end)
    gdf_res = gdf_res[gdf_res["built"] == 0]
    gdf_res["key"] = gdf_edges.apply(
        lambda df: tuple(
            (
                tuple(
                    [
                        float(val)
                        for val in df["from"]
                        .removeprefix("(")
                        .removesuffix(")")
                        .split(", ")
                    ]
                ),
                tuple(
                    [
                        float(val)
                        for val in df["to"]
                        .removeprefix("(")
                        .removesuffix(")")
                        .split(", ")
                    ]
                ),
                0,
            )
        ),
        axis=1,
    )
    gdf_res = gdf_res[["key", "level", "highway", "length", "geometry"]].rename(
        {"level": "bikenet_hierarchy", "highway": "road_hierarchy"}, axis=1
    )
    for met in [
        "betweenness",
        "closeness",
        "coverage",
        "directness",
        "dual_betweenness",
        "dual_closeness",
        "random",
        "road_hierarchy",
        "road_hierarchy_coverage",
        "road_hierarchy_directness",
    ]:
        gdf_order = gdf_res.copy()
        foldername = FOLDEROOT + end + f"/bs_{BUFF_SIZE}_{met}/"
        if met in ["coverage", "road_hierarchy", "random"]:
            foldername += met + "_000/"
        with open(foldername + "order_growth.json") as f:
            order_growth = json.load(f)
        order_growth = [
            tuple((tuple(val[0]), tuple(val[1]), val[2])) for val in order_growth
        ]
        with open(foldername + "metrics_growth.json") as f:
            metrics_growth = json.load(f)
        order_growth_dict = {val: idx + 1 for idx, val in enumerate(order_growth)}
        order_growth_dict_reversed = {
            val: idx + 1
            for idx, val in enumerate(
                tuple((val[1], val[0], val[2])) for val in order_growth
            )
        }
        gdf_order["order"] = gdf_order["key"].map(order_growth_dict)
        gdf_order["order_rev"] = gdf_order["key"].map(order_growth_dict_reversed)
        gdf_order["order"] = gdf_order["order"].fillna(gdf_order["order_rev"]).map(int)
        gdf_order = gdf_order.set_index("order").sort_index()
        gdf_order["total_length"] = metrics_growth["xx"][1:]
        gdf_order["total_coverage"] = metrics_growth["coverage"][1:]
        gdf_order["total_directness"] = metrics_growth["directness"][1:]
        gdf_order = gdf_order[
            [
                "bikenet_hierarchy",
                "road_hierarchy",
                "length",
                "total_length",
                "total_coverage",
                "total_directness",
                "geometry",
            ]
        ]
        gdf_order.to_file(folder_out + f"{met}_order_separated.geojson")